# BioBERT Fine-Tuning for Medical Named Entity Recognition (NER)

This notebook fine-tunes **dmis-lab/biobert-base-cased-v1.2** on prescription and lab-report NER data.

**Data files required** (place in the same folder as this notebook, or in `dataset_ner/`):
- `prescription_ner.json` (200 samples)
- `report_ner.json` (200 samples)

**Runtime:** Enable GPU in Colab (*Runtime → Change runtime type → GPU*). Training (~15 epochs) typically takes **15–30 minutes** on a T4 GPU.

In [1]:
# CELL 1: Install dependencies
# Run this cell first on Google Colab to install all required packages.

!pip install -q torch transformers datasets seqeval scikit-learn pandas tqdm

import torch

print("=" * 50)
print("GPU AVAILABILITY CHECK")
print("=" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")
else:
    print("WARNING: No GPU detected. Training will be very slow on CPU.")
    print("In Colab: Runtime → Change runtime type → GPU")
print("=" * 50)


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


GPU AVAILABILITY CHECK
PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU device      : NVIDIA GeForce RTX 4050 Laptop GPU
CUDA version    : 12.8


In [2]:
# CELL 2: Import all libraries

import json
import os
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

print("All libraries imported successfully.")
print(f"Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

All libraries imported successfully.
Using device: cuda


In [3]:
# CELL 3: Load and merge both JSON files
# Each file contains a list of {"tokens": [...], "labels": [...]} samples.

# Resolve data directory: same folder as notebook OR dataset_ner/ subfolder
if os.path.exists("prescription_ner.json"):
    DATA_DIR = "."
elif os.path.exists("dataset_ner/prescription_ner.json"):
    DATA_DIR = "dataset_ner"
else:
    raise FileNotFoundError(
        "Could not find prescription_ner.json. "
        "Place it in the notebook directory or in dataset_ner/"
    )

print(f"Loading data from: {DATA_DIR}/")

with open(os.path.join(DATA_DIR, "prescription_ner.json"), "r", encoding="utf-8") as f:
    prescription_data = json.load(f)

with open(os.path.join(DATA_DIR, "report_ner.json"), "r", encoding="utf-8") as f:
    report_data = json.load(f)

print(f"Prescription samples loaded : {len(prescription_data)}")
print(f"Report samples loaded       : {len(report_data)}")

# Tag each sample with its source before merging
for sample in prescription_data:
    sample["source"] = "prescription"
for sample in report_data:
    sample["source"] = "report"

# Merge into a single list (400 samples total)
all_data = prescription_data + report_data

print(f"\nTotal merged samples: {len(all_data)}")
print("\n--- Sample entry (first prescription) ---")
print(f"Source : {all_data[0]['source']}")
print(f"Tokens : {all_data[0]['tokens'][:15]} ...")
print(f"Labels : {all_data[0]['labels'][:15]} ...")

Loading data from: dataset_ner/
Prescription samples loaded : 200
Report samples loaded       : 200

Total merged samples: 400

--- Sample entry (first prescription) ---
Source : prescription
Tokens : ['Arjun', 'Mehta', 'MBBS', 'Ophthalmology', 'Fresher', 'Reg', 'Id', '7654821', 'Rx', 'ID', 'RX0567890', 'Neha', 'Kapoor', '27', 'Years'] ...
Labels : ['B-DOCTOR_NAME', 'I-DOCTOR_NAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-PATIENT_INFO', 'I-PATIENT_INFO', 'B-PATIENT_INFO', 'I-PATIENT_INFO'] ...


In [4]:
# CELL 4: Data cleaning and validation
# Replace any label NOT in the allowed BIO tag set with "O".

ALLOWED_LABELS = [
    "O",
    "B-DOCTOR_NAME", "I-DOCTOR_NAME",
    "B-PATIENT_INFO", "I-PATIENT_INFO",
    "B-DATE", "I-DATE",
    "B-COMPLAINT", "I-COMPLAINT",
    "B-DRUG_NAME", "I-DRUG_NAME",
    "B-DRUG_STRENGTH", "I-DRUG_STRENGTH",
    "B-DRUG_FORM", "I-DRUG_FORM",
    "B-DOSAGE", "I-DOSAGE",
    "B-FREQUENCY", "I-FREQUENCY",
    "B-DURATION", "I-DURATION",
    "B-ROUTE", "I-ROUTE",
    "B-INSTRUCTION", "I-INSTRUCTION",
    "B-DIAGNOSIS", "I-DIAGNOSIS",
    "B-VITALS", "I-VITALS",
    "B-ADDITIONAL_INFO", "I-ADDITIONAL_INFO",
    "B-REPORT_INFO", "I-REPORT_INFO",
    "B-TEST_NAME", "I-TEST_NAME",
    "B-TEST_VALUE", "I-TEST_VALUE",
    "B-TEST_UNIT", "I-TEST_UNIT",
    "B-REF_RANGE", "I-REF_RANGE",
    "B-TEST_STATUS", "I-TEST_STATUS",
    "B-INSTRUMENT", "I-INSTRUMENT",
    "B-INTERPRETATION", "I-INTERPRETATION",
]
ALLOWED_SET = set(ALLOWED_LABELS)

replaced_counter = Counter()
length_mismatches = 0

for sample in all_data:
    if len(sample["tokens"]) != len(sample["labels"]):
        length_mismatches += 1
        print(f"WARNING: length mismatch in sample (tokens={len(sample['tokens'])}, labels={len(sample['labels'])})")

    cleaned_labels = []
    for label in sample["labels"]:
        if label not in ALLOWED_SET:
            replaced_counter[label] += 1
            cleaned_labels.append("O")
        else:
            cleaned_labels.append(label)
    sample["labels"] = cleaned_labels

print(f"Length mismatches found: {length_mismatches}")
print(f"\nTotal labels replaced with 'O': {sum(replaced_counter.values())}")
if replaced_counter:
    print("\nReplaced label breakdown:")
    for label, count in replaced_counter.most_common():
        print(f"  {label:30s} -> O  ({count} times)")
else:
    print("No invalid labels found — data is clean.")

# Label distribution across all data
label_dist = Counter()
for sample in all_data:
    label_dist.update(sample["labels"])

print(f"\nLabel distribution ({len(label_dist)} unique labels):")
for label, count in sorted(label_dist.items(), key=lambda x: (-x[1], x[0])):
    print(f"  {label:30s} {count:6d}")

Length mismatches found: 0

Total labels replaced with 'O': 0
No invalid labels found — data is clean.

Label distribution (44 unique labels):
  O                                4302
  I-INTERPRETATION                 1604
  B-PATIENT_INFO                   1178
  I-ADDITIONAL_INFO                 893
  I-REPORT_INFO                     872
  I-PATIENT_INFO                    811
  I-REF_RANGE                       790
  I-VITALS                          710
  B-DRUG_NAME                       585
  B-FREQUENCY                       572
  I-INSTRUMENT                      563
  B-DURATION                        561
  B-DOSAGE                          552
  B-DRUG_STRENGTH                   550
  I-DURATION                        463
  I-FREQUENCY                       446
  I-INSTRUCTION                     442
  I-DOSAGE                          436
  I-DRUG_STRENGTH                   430
  B-TEST_NAME                       404
  B-TEST_VALUE                      404
  B-DATE         

In [5]:
# CELL 5: Build label mappings
# Collect unique labels, sort with O first then alphabetically.

unique_labels = set()
for sample in all_data:
    unique_labels.update(sample["labels"])

sorted_labels = sorted(unique_labels, key=lambda x: (x != "O", x))

label2id = {label: idx for idx, label in enumerate(sorted_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print(f"Total unique labels: {len(sorted_labels)}")
print("\nlabel2id mapping:")
for label, idx in label2id.items():
    print(f"  {idx:3d} -> {label}")

Total unique labels: 44

label2id mapping:
    0 -> O
    1 -> B-ADDITIONAL_INFO
    2 -> B-COMPLAINT
    3 -> B-DATE
    4 -> B-DIAGNOSIS
    5 -> B-DOCTOR_NAME
    6 -> B-DOSAGE
    7 -> B-DRUG_FORM
    8 -> B-DRUG_NAME
    9 -> B-DRUG_STRENGTH
   10 -> B-DURATION
   11 -> B-FREQUENCY
   12 -> B-INSTRUCTION
   13 -> B-INSTRUMENT
   14 -> B-INTERPRETATION
   15 -> B-PATIENT_INFO
   16 -> B-REF_RANGE
   17 -> B-REPORT_INFO
   18 -> B-ROUTE
   19 -> B-TEST_NAME
   20 -> B-TEST_STATUS
   21 -> B-TEST_UNIT
   22 -> B-TEST_VALUE
   23 -> B-VITALS
   24 -> I-ADDITIONAL_INFO
   25 -> I-COMPLAINT
   26 -> I-DATE
   27 -> I-DIAGNOSIS
   28 -> I-DOCTOR_NAME
   29 -> I-DOSAGE
   30 -> I-DRUG_NAME
   31 -> I-DRUG_STRENGTH
   32 -> I-DURATION
   33 -> I-FREQUENCY
   34 -> I-INSTRUCTION
   35 -> I-INSTRUMENT
   36 -> I-INTERPRETATION
   37 -> I-PATIENT_INFO
   38 -> I-REF_RANGE
   39 -> I-REPORT_INFO
   40 -> I-ROUTE
   41 -> I-TEST_NAME
   42 -> I-TEST_UNIT
   43 -> I-VITALS


In [6]:
# CELL 6: Train / Validation / Test split (80 / 10 / 10)
# Stratified by source: split prescription and report samples separately,
# then combine so each split contains both document types.

RANDOM_SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.8, 0.1, 0.1

prescription_samples = [s for s in all_data if s["source"] == "prescription"]
report_samples = [s for s in all_data if s["source"] == "report"]


def stratified_split(samples, seed=RANDOM_SEED):
    """Split one source group into train / val / test."""
    train, temp = train_test_split(
        samples,
        test_size=VAL_RATIO + TEST_RATIO,
        random_state=seed,
    )
    relative_test = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
    val, test = train_test_split(
        temp,
        test_size=relative_test,
        random_state=seed,
    )
    return train, val, test


p_train, p_val, p_test = stratified_split(prescription_samples)
r_train, r_val, r_test = stratified_split(report_samples)

train_data = p_train + r_train
val_data = p_val + r_val
test_data = p_test + r_test


def print_split_stats(name, data):
    sources = Counter(s["source"] for s in data)
    print(f"  {name:6s}: {len(data):3d} total | prescription={sources['prescription']}, report={sources['report']}")


print("Split counts:")
print_split_stats("Train", train_data)
print_split_stats("Val", val_data)
print_split_stats("Test", test_data)
print(f"\nTotal: {len(train_data) + len(val_data) + len(test_data)} samples")

Split counts:
  Train : 320 total | prescription=160, report=160
  Val   :  40 total | prescription=20, report=20
  Test  :  40 total | prescription=20, report=20

Total: 400 samples


In [7]:
# CELL 7: Load BioBERT tokenizer and model
# Using dmis-lab/biobert-base-cased-v1.2 (NOT v1.1)

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

print(f"Loading tokenizer from {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model from {MODEL_NAME} ...")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

param_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel loaded successfully.")
print(f"Total parameters     : {param_count:,}")
print(f"Trainable parameters : {trainable_count:,}")
print(f"Number of NER labels : {len(label2id)}")

Loading tokenizer from dmis-lab/biobert-base-cased-v1.2 ...


Loading model from dmis-lab/biobert-base-cased-v1.2 ...


[transformers] You passed `num_labels=44` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	


Model loaded successfully.
Total parameters     : 107,753,516
Trainable parameters : 107,753,516
Number of NER labels : 44


In [8]:
# CELL 8: Tokenization with subword alignment
# Critical step: align word-level BIO labels to BioBERT subword tokens.
#   - First subword of a word  -> original label
#   - Further subwords of a word -> B- converted to I-
#   - Special/padding tokens -> -100 (ignored in loss)


def tokenize_and_align_labels(examples):
    """Tokenize pre-split words and align NER labels to subword tokens."""
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=256,
        padding="max_length",
    )

    all_labels = []
    unknown_labels = set()
    
    for i, word_labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                # Special token — ignore in loss computation
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword token of this word
                original_label = word_labels[word_idx]
                if original_label not in label2id:
                    unknown_labels.add(original_label)
                    label_ids.append(label2id["O"])  # fallback to "O"
                else:
                    label_ids.append(label2id[original_label])
            else:
                # Continuation subword — convert B- to I-
                label_name = word_labels[word_idx]
                if label_name.startswith("B-"):
                    label_name = "I-" + label_name[2:]
                
                if label_name not in label2id:
                    unknown_labels.add(label_name)
                    label_ids.append(label2id["O"])  # fallback to "O"
                else:
                    label_ids.append(label2id[label_name])
            previous_word_idx = word_idx

        all_labels.append(label_ids)
    
    # Print warning if unknown labels encountered
    if unknown_labels:
        print(f"WARNING: Unknown labels encountered in batch and replaced with 'O': {unknown_labels}")
    
    tokenized["labels"] = all_labels
    return tokenized


def to_hf_dataset(samples):
    """Convert list of dicts to HuggingFace Dataset (drop source tag)."""
    return Dataset.from_dict({
        "tokens": [s["tokens"] for s in samples],
        "labels": [s["labels"] for s in samples],
    })


print("Converting splits to HuggingFace Dataset format ...")
raw_train = to_hf_dataset(train_data)
raw_val = to_hf_dataset(val_data)
raw_test = to_hf_dataset(test_data)

print("Tokenizing and aligning labels (this may take a minute) ...")
train_dataset = raw_train.map(tokenize_and_align_labels, batched=True, remove_columns=raw_train.column_names)
val_dataset = raw_val.map(tokenize_and_align_labels, batched=True, remove_columns=raw_val.column_names)
test_dataset = raw_test.map(tokenize_and_align_labels, batched=True, remove_columns=raw_test.column_names)

print(f"\nTokenized dataset sizes: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")

# Show one tokenized example for verification
example_idx = 0
input_ids = train_dataset[example_idx]["input_ids"]
aligned_labels = train_dataset[example_idx]["labels"]
word_ids = train_dataset[example_idx].get("word_ids")  # may not be stored

# Recompute word_ids for display
enc = tokenizer(train_data[example_idx]["tokens"], is_split_into_words=True, truncation=True, max_length=256, padding="max_length")
word_ids_display = enc.word_ids()

print("\n--- Tokenized example (first training sample) ---")
print(f"{'Idx':>4} {'Token':20s} {'WordID':>7} {'LabelID':>8} {'Label':20s}")
print("-" * 65)
for idx, (tid, lid, wid) in enumerate(zip(input_ids[:30], aligned_labels[:30], word_ids_display[:30])):
    token_str = tokenizer.convert_ids_to_tokens(tid)
    label_str = id2label[lid] if lid != -100 else "[IGNORE]"
    print(f"{idx:4d} {token_str:20s} {str(wid):>7} {lid:8d} {label_str:20s}")
print("... (truncated display)")

Converting splits to HuggingFace Dataset format ...
Tokenizing and aligning labels (this may take a minute) ...


Map:   0%|          | 0/320 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Tokenized dataset sizes: train=320, val=40, test=40

--- Tokenized example (first training sample) ---
 Idx Token                 WordID  LabelID Label               
-----------------------------------------------------------------
   0 [CLS]                   None     -100 [IGNORE]            
   1 ka                         0        5 B-DOCTOR_NAME       
   2 ##l                        0       28 I-DOCTOR_NAME       
   3 ##pan                      0       28 I-DOCTOR_NAME       
   4 ##a                        0       28 I-DOCTOR_NAME       
   5 b                          1       28 I-DOCTOR_NAME       
   6 ##hat                      1       28 I-DOCTOR_NAME       
   7 ##t                        1       28 I-DOCTOR_NAME       
   8 m                          2        0 O                   
   9 ##d                        2        0 O                   
  10 f                          3       15 B-PATIENT_INFO      
  11 ##ai                       3       37 I-PATIENT_INFO     

In [9]:
# CELL 9: Define compute_metrics function
# Uses seqeval for entity-level precision, recall, and F1.


def compute_metrics(eval_pred):
    """Compute micro-averaged and per-entity NER metrics."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Convert IDs to label strings, skipping ignored positions (-100)
    true_labels = []
    pred_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        true_seq = []
        pred_seq_clean = []
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue
            true_seq.append(id2label[label_id])
            pred_seq_clean.append(id2label[pred_id])
        true_labels.append(true_seq)
        pred_labels.append(pred_seq_clean)

    return {
        "precision": precision_score(true_labels, pred_labels),
        "recall": recall_score(true_labels, pred_labels),
        "f1": f1_score(true_labels, pred_labels),
    }


print("compute_metrics function defined.")

compute_metrics function defined.


In [13]:
# CELL 10: Training configuration
# Set up TrainingArguments, DataCollator, and Trainer.

OUTPUT_DIR = "./biobert-medical-ner"
USE_FP16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="best",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    warmup_steps=50,
    logging_steps=10,
    fp16=USE_FP16,
    report_to="none",  # disable wandb/tensorboard on Colab
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

steps_per_epoch = len(train_dataset) // training_args.per_device_train_batch_size
total_steps = steps_per_epoch * training_args.num_train_epochs

print("Trainer initialized.")
print(f"  Output directory : {OUTPUT_DIR}")
print(f"  Epochs           : {training_args.num_train_epochs}")
print(f"  Batch size       : {training_args.per_device_train_batch_size}")
print(f"  Learning rate    : {training_args.learning_rate}")
print(f"  FP16 enabled     : {USE_FP16}")
print(f"  Steps per epoch  : ~{steps_per_epoch}")
print(f"  Total steps      : ~{total_steps}")
print(f"\nEstimated training time: 15–30 min on T4 GPU, 2–4 hrs on CPU.")

Trainer initialized.
  Output directory : ./biobert-medical-ner
  Epochs           : 15
  Batch size       : 16
  Learning rate    : 2e-05
  FP16 enabled     : True
  Steps per epoch  : ~20
  Total steps      : ~300

Estimated training time: 15–30 min on T4 GPU, 2–4 hrs on CPU.


In [15]:
# CELL 11: Train the model
# This cell will take the longest — monitor loss and F1 in the output.

import time

print("Starting training ...")
print("=" * 50)
start_time = time.time()

train_result = trainer.train()

elapsed = time.time() - start_time
minutes, seconds = divmod(int(elapsed), 60)

print("=" * 50)
print("TRAINING COMPLETE")
print(f"Training time: {minutes}m {seconds}s")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Global step  : {train_result.global_step}")
print("=" * 50)

Starting training ...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.226761,0.344179,0.791014,0.878190,0.832325


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:745] . open file failed with error code: 1224

In [ ]:
# CELL 11B: Remove intermediate Trainer output directory

import shutil

if os.path.exists("./biobert-medical-ner"):
    shutil.rmtree("./biobert-medical-ner")

In [ ]:
# CELL 12: Evaluate on test set

print("Evaluating on held-out test set ...")
test_results = trainer.evaluate(test_dataset)

print("\n" + "=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"  Precision : {test_results['eval_precision']:.4f}")
print(f"  Recall    : {test_results['eval_recall']:.4f}")
print(f"  F1 Score  : {test_results['eval_f1']:.4f}")
print(f"  Loss      : {test_results['eval_loss']:.4f}")
print("=" * 50)

Evaluating on held-out test set ...


Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.024932,0.081793,15,0.950820,0.976684,0.963578



TEST SET RESULTS
  Precision : 0.9508
  Recall    : 0.9767
  F1 Score  : 0.9636
  Loss      : 0.0818


In [ ]:
# CELL 13: Detailed per-entity evaluation
# Shows precision, recall, F1, and support for EACH entity type.

print("Running predictions on test set for per-entity analysis ...")
predictions_output = trainer.predict(test_dataset)
pred_logits = predictions_output.predictions
true_label_ids = predictions_output.label_ids

pred_ids = np.argmax(pred_logits, axis=-1)

true_labels_list = []
pred_labels_list = []

for pred_seq, label_seq in zip(pred_ids, true_label_ids):
    true_seq, pred_seq_clean = [], []
    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id == -100:
            continue
        true_seq.append(id2label[label_id])
        pred_seq_clean.append(id2label[pred_id])
    true_labels_list.append(true_seq)
    pred_labels_list.append(pred_seq_clean)

print("\n" + "=" * 70)
print("PER-ENTITY CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(true_labels_list, pred_labels_list, digits=4))
print("=" * 70)

Running predictions on test set for per-entity analysis ...



PER-ENTITY CLASSIFICATION REPORT
                 precision    recall  f1-score   support

ADDITIONAL_INFO     1.0000    1.0000    1.0000        13
      COMPLAINT     0.7500    0.8333    0.7895        18
           DATE     0.9286    0.9750    0.9512        40
      DIAGNOSIS     1.0000    1.0000    1.0000        17
    DOCTOR_NAME     0.9756    1.0000    0.9877        40
         DOSAGE     0.9574    0.9574    0.9574        47
      DRUG_FORM     0.9062    0.9667    0.9355        30
      DRUG_NAME     1.0000    1.0000    1.0000        49
  DRUG_STRENGTH     1.0000    0.9787    0.9892        47
       DURATION     1.0000    1.0000    1.0000        45
      FREQUENCY     0.9792    1.0000    0.9895        47
    INSTRUCTION     1.0000    1.0000    1.0000        34
     INSTRUMENT     0.9286    1.0000    0.9630        13
 INTERPRETATION     0.8462    0.8462    0.8462        13
   PATIENT_INFO     0.9912    1.0000    0.9956       113
      REF_RANGE     1.0000    1.0000    1.0000       

In [ ]:
# CELL 14: Confusion analysis
# Find the most common gold-label -> predicted-label errors (excluding correct predictions).

error_counter = Counter()

for true_seq, pred_seq in zip(true_labels_list, pred_labels_list):
    for true_lbl, pred_lbl in zip(true_seq, pred_seq):
        if true_lbl != pred_lbl:
            error_counter[(true_lbl, pred_lbl)] += 1

print(f"Total prediction errors: {sum(error_counter.values())}")
print(f"Unique error types    : {len(error_counter)}")
print("\nTop 10 most common prediction errors:")
print(f"{'Gold Label':30s} {'Predicted As':30s} {'Count':>6}")
print("-" * 70)

for (gold, pred), count in error_counter.most_common(10):
    print(f"{gold:30s} {pred:30s} {count:6d}")

# Entity-type level confusion (collapse B-/I- to entity name)
def to_entity_type(label):
    if label == "O":
        return "O"
    return label[2:]  # strip B- or I-

entity_error_counter = Counter()
for true_seq, pred_seq in zip(true_labels_list, pred_labels_list):
    for true_lbl, pred_lbl in zip(true_seq, pred_seq):
        if true_lbl != pred_lbl and true_lbl != "O":
            entity_error_counter[(to_entity_type(true_lbl), to_entity_type(pred_lbl))] += 1

print("\nTop 10 entity-type confusions (gold entity -> predicted entity):")
print(f"{'Gold Entity':25s} {'Predicted Entity':25s} {'Count':>6}")
print("-" * 60)
for (gold_ent, pred_ent), count in entity_error_counter.most_common(10):
    print(f"{gold_ent:25s} {pred_ent:25s} {count:6d}")

Total prediction errors: 76
Unique error types    : 33

Top 10 most common prediction errors:
Gold Label                     Predicted As                    Count
----------------------------------------------------------------------
O                              I-TEST_NAME                        14
O                              I-COMPLAINT                         7
I-COMPLAINT                    O                                   5
O                              I-VITALS                            5
I-REPORT_INFO                  O                                   4
O                              B-TEST_NAME                         3
O                              I-DATE                              3
I-REPORT_INFO                  I-INSTRUMENT                        3
B-COMPLAINT                    O                                   2
O                              B-DRUG_FORM                         2

Top 10 entity-type confusions (gold entity -> predicted entity):
Gold Entit

In [ ]:
# CELL 15: Inference demo — predict_entities function
# Tokenizes raw text, runs the model, and reconstructs entities from BIO tags.

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def predict_entities(text, model=model, tokenizer=tokenizer, id2label=id2label):
    """
    Extract named entities from raw text.

    Returns list of dicts: [{"entity": "DRUG_NAME", "text": "Paracetamol"}, ...]
    """
    # Simple whitespace tokenization (matches training data format)
    words = text.split()

    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True,
    )
    word_ids = encoding.word_ids(batch_index=0)
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().tolist()

    # Collect one label per word (use first subword token's prediction)
    word_labels = ["O"] * len(words)
    previous_word_idx = None
    for idx, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != previous_word_idx:
            word_labels[word_idx] = id2label[predictions[idx]]
        previous_word_idx = word_idx

    # Reconstruct entities from BIO tags
    entities = []
    current_entity = None
    current_tokens = []

    for word, label in zip(words, word_labels):
        if label.startswith("B-"):
            if current_entity is not None:
                entities.append({"entity": current_entity, "text": " ".join(current_tokens)})
            current_entity = label[2:]
            current_tokens = [word]
        elif label.startswith("I-") and current_entity == label[2:]:
            current_tokens.append(word)
        else:
            if current_entity is not None:
                entities.append({"entity": current_entity, "text": " ".join(current_tokens)})
                current_entity = None
                current_tokens = []

    if current_entity is not None:
        entities.append({"entity": current_entity, "text": " ".join(current_tokens)})

    return entities


# Store Cell 15 outputs for verification in Cell 17
SAMPLE_TEXTS = [
  "Dr. Ramesh Gupta prescribed Paracetamol 650 mg Tablet 1-0-1 after food for 5 days",
  "Dr. Anita Verma Hemoglobin 12.5 g/dL 13.0 - 17.0 Low Total RBC count 5.2 mill/cumm 4.5 - 5.5",
]

cell15_results = {}

for i, sample_text in enumerate(SAMPLE_TEXTS, 1):
    print(f"\n{'=' * 60}")
    print(f"Sample {i}: {sample_text}")
    print("=" * 60)
    entities = predict_entities(sample_text)
    cell15_results[sample_text] = entities
    if entities:
        for ent in entities:
            print(f"  [{ent['entity']:20s}] {ent['text']}")
    else:
        print("  No entities detected.")


Sample 1: Dr. Ramesh Gupta prescribed Paracetamol 650 mg Tablet 1-0-1 after food for 5 days
  [DOCTOR_NAME         ] Ramesh Gupta
  [DRUG_NAME           ] Paracetamol
  [DRUG_STRENGTH       ] 650 mg
  [DRUG_FORM           ] Tablet
  [FREQUENCY           ] 1-0-1
  [INSTRUCTION         ] after food for
  [DURATION            ] 5 days

Sample 2: Dr. Anita Verma Hemoglobin 12.5 g/dL 13.0 - 17.0 Low Total RBC count 5.2 mill/cumm 4.5 - 5.5
  [DOCTOR_NAME         ] Anita Verma
  [TEST_NAME           ] Hemoglobin
  [TEST_VALUE          ] 12.5
  [TEST_UNIT           ] g/dL
  [REF_RANGE           ] 13.0 - 17.0
  [TEST_STATUS         ] Low
  [TEST_NAME           ] Total RBC count
  [TEST_VALUE          ] 5.2
  [TEST_UNIT           ] mill/cumm
  [REF_RANGE           ] 4.5 - 5.5


In [ ]:
# CELL 16: Save the model and all artifacts for FastAPI inference

SAVE_DIR = "./biobert-medical-ner-final"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save model and tokenizer
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to   : {SAVE_DIR}")
print(f"Tokenizer saved to: {SAVE_DIR}")

# Save label mappings
label_mappings = {
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
}
mappings_path = os.path.join(SAVE_DIR, "label_mappings.json")
with open(mappings_path, "w", encoding="utf-8") as f:
    json.dump(label_mappings, f, indent=2)
print(f"Label mappings  : {mappings_path}")
print("\nAll saved files:")
for root, dirs, files in os.walk(SAVE_DIR):
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  {fpath}  ({size_kb:.1f} KB)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to   : ./biobert-medical-ner-final
Tokenizer saved to: ./biobert-medical-ner-final
Label mappings  : ./biobert-medical-ner-final\label_mappings.json

All saved files:
  ./biobert-medical-ner-final\config.json  (3.1 KB)
  ./biobert-medical-ner-final\label_mappings.json  (2.2 KB)
  ./biobert-medical-ner-final\model.safetensors  (420935.0 KB)
  ./biobert-medical-ner-final\tokenizer.json  (653.5 KB)
  ./biobert-medical-ner-final\tokenizer_config.json  (0.4 KB)


In [ ]:
# CELL 17: Verify saved model loads correctly and reproduces Cell 15 outputs

VERIFY_DIR = "./biobert-medical-ner-final"

print(f"Loading model from {VERIFY_DIR} ...")
loaded_tokenizer = AutoTokenizer.from_pretrained(VERIFY_DIR)

with open(os.path.join(VERIFY_DIR, "label_mappings.json"), "r", encoding="utf-8") as f:
    loaded_mappings = json.load(f)
loaded_label2id = loaded_mappings["label2id"]
loaded_id2label = {int(k): v for k, v in loaded_mappings["id2label"].items()}

loaded_model = AutoModelForTokenClassification.from_pretrained(
    VERIFY_DIR,
    id2label=loaded_id2label,
    label2id=loaded_label2id,
)
loaded_model.eval()
loaded_model.to(device)

print("Model, tokenizer, and label mappings loaded successfully.\n")

all_match = True
for sample_text in SAMPLE_TEXTS:
    original = cell15_results[sample_text]
    reloaded = predict_entities(
        sample_text,
        model=loaded_model,
        tokenizer=loaded_tokenizer,
        id2label=loaded_id2label,
    )
    match = original == reloaded
    all_match = all_match and match

    print(f"Text   : {sample_text[:60]}...")
    print(f"Match  : {'YES' if match else 'NO'}")
    if not match:
        print(f"  Cell 15 : {original}")
        print(f"  Cell 17 : {reloaded}")
    print()

if all_match:
    print("Model saved and verified successfully!")
else:
    print("WARNING: Reloaded model outputs differ from Cell 15. Check save/load paths.")

Loading model from ./biobert-medical-ner-final ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model, tokenizer, and label mappings loaded successfully.

Text   : Dr. Ramesh Gupta prescribed Paracetamol 650 mg Tablet 1-0-1 ...
Match  : YES

Text   : Dr. Anita Verma Hemoglobin 12.5 g/dL 13.0 - 17.0 Low Total R...
Match  : YES

Model saved and verified successfully!
